In [1]:
import os
import pandas as pd
import numpy as np


# ============================================================
# 1. PROJECT PATHS
# ============================================================

BASE_DIR = os.getcwd()

# If running from notebooks/ folder
if os.path.basename(BASE_DIR).lower() == "notebooks":
    BASE_DIR = os.path.dirname(BASE_DIR)

INPUT_FILE = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "pune_cpcb_combined_raw.csv.gz"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "pune_cpcb_cleaned_hourly.csv"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 70)
print("PUNE CPCB DATA CLEANING")
print("=" * 70)

print("\nLoading:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE)

print("\nOriginal shape:", df.shape)


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
)

print("\nColumns loaded:")
print(df.columns.tolist())


# ============================================================
# 4. CREATE STATION NAME
# ============================================================

if "station_name" not in df.columns:

    if "Station Name" in df.columns:

        df["station_name"] = (
            df["Station Name"]
            .astype(str)
            .str.strip()
        )

    else:

        raise KeyError(
            "station_name / Station Name column not found."
        )

else:

    df["station_name"] = (
        df["station_name"]
        .astype(str)
        .str.strip()
    )


# ============================================================
# 5. FIND TIMESTAMP COLUMN
# ============================================================

timestamp_candidates = [
    "timestamp",
    "Timestamp",
    "Date",
    "date",
    "datetime",
    "Datetime",
    "Date & Time",
    "From Date",
    "From date"
]

timestamp_column = None

for column in timestamp_candidates:

    if column in df.columns:

        timestamp_column = column
        break


if timestamp_column is None:

    raise KeyError(
        "Timestamp column was not found. "
        "Check df.columns.tolist()"
    )


print("\nTimestamp column:", timestamp_column)


# ============================================================
# 6. STANDARDIZE TIMESTAMP
# ============================================================

df["timestamp"] = pd.to_datetime(
    df[timestamp_column],
    errors="coerce"
)

# Remove rows with invalid timestamps
invalid_timestamp = df["timestamp"].isna().sum()

print(
    "Invalid timestamps removed:",
    invalid_timestamp
)

df = df.dropna(
    subset=["timestamp"]
).copy()


# ============================================================
# 7. REMOVE UNNECESSARY COLUMNS
# ============================================================

drop_columns = [
    "VWS (m/s)",
    "O Xylene (µg/m³)",
    "BP (mmHg)",
    "Xylene (µg/m³)",
    "NH3 (µg/m³)",
    "SR (W/mt2)"
]

existing_drop_columns = [
    col for col in drop_columns
    if col in df.columns
]

df = df.drop(
    columns=existing_drop_columns
)

print("\nDropped columns:")
print(existing_drop_columns)


# ============================================================
# 8. RENAME IMPORTANT COLUMNS
# ============================================================

rename_columns = {
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "SO2 (µg/m³)": "so2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "ozone",
    "AT (°C)": "temperature",
    "RH (%)": "humidity",
    "WS (m/s)": "wind_speed",
    "WD (deg)": "wind_direction",
    "RF (mm)": "rainfall"
}

df = df.rename(
    columns=rename_columns
)


# ============================================================
# 9. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "ozone",
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "rainfall"
]

numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 10. REMOVE EXACT DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nExact duplicates removed:", before - after)


# ============================================================
# 11. REMOVE DUPLICATE STATION + TIMESTAMP
# ============================================================

duplicate_count = df.duplicated(
    subset=["station_name", "timestamp"]
).sum()

print(
    "Duplicate station-timestamp rows:",
    duplicate_count
)

if duplicate_count > 0:

    # Keep the first valid record
    df = (
        df.sort_values(
            ["station_name", "timestamp"]
        )
        .drop_duplicates(
            subset=["station_name", "timestamp"],
            keep="first"
        )
    )


# ============================================================
# 12. SORT DATA
# ============================================================

df = df.sort_values(
    ["station_name", "timestamp"]
).reset_index(drop=True)


# ============================================================
# 13. REMOVE IMPOSSIBLE SENSOR VALUES
# ============================================================

# PM2.5 cannot be negative
if "pm25" in df.columns:

    df.loc[
        df["pm25"] < 0,
        "pm25"
    ] = np.nan


# PM10 cannot be negative
if "pm10" in df.columns:

    df.loc[
        df["pm10"] < 0,
        "pm10"
    ] = np.nan


# Other pollution values cannot be negative
for column in [
    "no2",
    "so2",
    "co",
    "ozone"
]:

    if column in df.columns:

        df.loc[
            df[column] < 0,
            column
        ] = np.nan


# Humidity should be 0–100
if "humidity" in df.columns:

    df.loc[
        (df["humidity"] < 0) |
        (df["humidity"] > 100),
        "humidity"
    ] = np.nan


# Wind speed cannot be negative
if "wind_speed" in df.columns:

    df.loc[
        df["wind_speed"] < 0,
        "wind_speed"
    ] = np.nan


# ============================================================
# 14. HOURLY AGGREGATION
# ============================================================

print("\n" + "=" * 70)
print("CONVERTING 15-MINUTE DATA TO HOURLY DATA")
print("=" * 70)

# Keep only columns that actually exist
hourly_columns = [
    col for col in numeric_columns
    if col in df.columns
]

df_hourly = (
    df.set_index("timestamp")
      .groupby("station_name")[hourly_columns]
      .resample("1h")
      .mean()
      .reset_index()
)


# ============================================================
# 15. REMOVE HOURS WITHOUT PM2.5
# ============================================================

if "pm25" in df_hourly.columns:

    before = len(df_hourly)

    df_hourly = df_hourly.dropna(
        subset=["pm25"]
    )

    after = len(df_hourly)

    print(
        "\nHourly rows removed because PM2.5 was missing:",
        before - after
    )


# ============================================================
# 16. FILL FEATURE MISSING VALUES STATION-WISE
# ============================================================

feature_columns = [
    "pm10",
    "no2",
    "so2",
    "co",
    "ozone",
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "rainfall"
]

feature_columns = [
    col for col in feature_columns
    if col in df_hourly.columns
]


for column in feature_columns:

    df_hourly[column] = (
        df_hourly
        .groupby("station_name")[column]
        .transform(
            lambda x: x.ffill().bfill()
        )
    )


# ============================================================
# 17. CHECK REMAINING MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("REMAINING MISSING VALUES")
print("=" * 70)

missing = (
    df_hourly
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing)


# ============================================================
# 18. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print("\nShape:")
print(df_hourly.shape)

print("\nStations:")
print(
    df_hourly["station_name"]
    .value_counts()
)

print("\nDate range:")

print(
    "Start:",
    df_hourly["timestamp"].min()
)

print(
    "End:",
    df_hourly["timestamp"].max()
)


# ============================================================
# 19. SAVE CLEANED DATA
# ============================================================

df_hourly.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("✅ DATA CLEANING COMPLETED")
print("=" * 70)

print("\nSaved file:")
print(OUTPUT_FILE)

print("\nFinal shape:")
print(df_hourly.shape)

PUNE CPCB DATA CLEANING

Loading:
c:\final year project\AI-POLLUTION-SYSTEM\data\processed\pune_cpcb_combined_raw.csv.gz

Original shape: (299145, 30)

Columns loaded:
['Station ID', 'State', 'City', 'Station Name', 'Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)', 'station_name']

Timestamp column: Timestamp
Invalid timestamps removed: 5

Dropped columns:
['VWS (m/s)', 'O Xylene (µg/m³)', 'BP (mmHg)', 'Xylene (µg/m³)', 'NH3 (µg/m³)', 'SR (W/mt2)']

Exact duplicates removed: 0
Duplicate station-timestamp rows: 0

CONVERTING 15-MINUTE DATA TO HOURLY DATA

Hourly rows removed because PM2.5 was missing: 19465

REMAINING MISSING VALUES
temperature       54.967999
ra